In [2]:
bc_umi_read_id_path="E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp99/BSTMAtrix/01.fastq2BcUmi/exp99_28Feb2025.bc_umi_read.tsv.id"
bc_umi_read_path="E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp99/BSTMAtrix/01.fastq2BcUmi/exp99_28Feb2025.bc_umi_read.tsv"
subdata_barcodes_path="E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp99/BSTMAtrix/05.AllheStat/BSTViewer_project/subdata/L1_heAuto/barcodes.tsv.gz"
barcode_path="E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp99/BSTMAtrix/exp99_28Feb2025/barcode.tsv"
barcode_cluster_path="E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp99/BSTMAtrix/05.AllheStat/BSTViewer_project/level_matrix/level_1/barcodes_cluster.tsv.gz"
input_bam_path="E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp99/BSTMAtrix/02.Umi2Gene/exp99_28Feb2025Aligned.out.bam"
output_bam_path="E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp99/bam_output/exp99_tagged.bam"
prefix="L1"

In [ ]:
# FASTQ total reads: 522,855,800 (100%)
# Input BAM (Aligned.out.bam): 593,902,518. Too much, likely due to multimapping. 
# Output BAM (exp99_tagged.bam): 431,446,444 (82.5%). Too little. Perl script drops reads for which the read name was not in bc_umi_read.tsv.id or if its barcode is not in barcodes_cluster.tsv.gz 

# FASTQ has 522M reads, but not all reads have been mapped, contain valid barcodes, etc.

In [ ]:
# According to the report file:
# Number of Reads: 522,855,800 (100%)
# Valid Barcodes: 479,798,105 (91.76%)
# Valid UMIs: 520,293,938 (99.51%)
# Valid Reads: 479,767,879 (91.76%)

# Reads Mapped to Genome: 454,413,976 (86.91%)
# Reads Mapped Confidently to Genome: 420,846,633 (80.49%)
# Reads Mapped Confidently to Intergenic Regions: 42,037,606 (8.04%)
# Reads Mapped Confidently to Intronic Regions: 35,867,907 (6.86%)
# Reads Mapped Confidently to Exonic Regions: 342,941,119 (65.59%)
# Reads Mapped Confidently to Transcriptome: 367,515,341 (70.29%) 

In [ ]:
# BAM output:
# CR:Z:L1_1779019 CB:Z:L1_1779019-1       UR:Z:V2Abc1_230-V2Abc2_271-V2Abc3_228_ATCATGCTCATA UB:Z:V2Abc1_230-V2Abc2_271-V2Abc3_228_ATCATGCTCATA  
# CR: Spatial barcode
# CB: Spatial barcode with -1 (basically the same as CR)
# UR: UMI
# UB: UMI (exactly the same as UR)

In [3]:
import pandas as pd

In [5]:
bc_umi_read = pd.read_csv(bc_umi_read_path, sep='\t', header=None)
bc_umi_read # Spatial barcount, UMI, counts?

,0,1,2
0,V2Bbc1_3-V2Abc2_368-V2Abc3_152,GCCTCTACTCAT,4
1,V2Bbc1_6-V2Bbc2_46-V2Abc3_379,TATCTCTTCG,3
2,V2Bbc1_6-V2Abc2_161-V2Bbc3_69,AGTGTGGAGAG,1
3,V2Bbc1_6-V2Abc2_161-V2Bbc3_69,ATTGACGCTAT,1
4,V2Bbc1_1-V2Abc2_209-V2Abc3_35,AACTTGAATAGA,2
...,...,...,...
236281986,V2Bbc1_59-V2Bbc2_13-V2Abc3_254,TGACCAGCGTGT,5
236281987,V2Abc1_60-V2Abc2_279-V2Abc3_358,TAATCCATCGGG,1
236281988,V2Abc1_60-V2Abc2_279-V2Abc3_358,ATGTGCCGATTA,1
236281989,V2Abc1_60-V2Abc2_279-V2Abc3_358,TCAACGCTTATC,1


In [9]:
bc_umi_read[2].sum() # Sum is 479,767,879.

479767879

In [17]:
bc_umi_read[0].nunique() # 10,774,536 unique spatial barcodes

10774536

In [16]:
bc_umi_read[1].nunique() # 22,117,756 unique UMIs

22117756

In [7]:
with open(bc_umi_read_id_path) as f:
    print("=== First 5 rows ===")
    for i, line in enumerate(f):
        print(line)
        if i >= 4:  # stop after 5 rows
            break

# Count rows efficiently without loading into memory
with open(bc_umi_read_id_path) as f:
    row_count = sum(1 for _ in f)

print(f"\nTotal rows: {row_count:,}")

=== First 5 rows ===
V2Bbc1_3-V2Abc2_368-V2Abc3_152	GCCTCTACTCAT	16657454	51124965	263627706	48397223

V2Bbc1_6-V2Bbc2_46-V2Abc3_379	TATCTCTTCG	47081548	113688040	125202339

V2Bbc1_6-V2Abc2_161-V2Bbc3_69	AGTGTGGAGAG	83683396

V2Bbc1_6-V2Abc2_161-V2Bbc3_69	ATTGACGCTAT	262555551

V2Bbc1_1-V2Abc2_209-V2Abc3_35	AACTTGAATAGA	194302329	207721981


Total rows: 236,281,991


In [ ]:
subdata_barcodes = pd.read_csv(subdata_barcodes_path, sep='\t', header=None)
subdata_barcodes # 1,960,273 spots. In total, there should be 3,482,159 spots in the capture area. This is the subset that is present in the tissue (according to BSTMAtrix)
# 3,295,405 were present in the capture area that had at least 2 UMI (according to harpy analysis)

,0
0,L1_289228
1,L1_289229
2,L1_289230
3,L1_290157
4,L1_290158
...,...
1960268,L1_3819759
1960269,L1_3819760
1960270,L1_3819761
1960271,L1_3821167


In [ ]:
barcode = pd.read_csv(barcode_path, sep='\t', header=None)
barcode # 3,482,159 spots. This matches the number of spots expected in the capture area (according to harpy analysis, number of spots included in table).

,0
0,V2Bbc1_359-V2Abc2_4-V2Bbc3_108
1,V2Abc1_220-V2Bbc2_349-V2Abc3_56
2,V2Abc1_353-V2Bbc2_379-V2Abc3_375
3,V2Bbc1_86-V2Abc2_263-V2Bbc3_14
4,V2Bbc1_186-V2Bbc2_104-V2Abc3_223
...,...
3482154,V2Bbc1_71-V2Bbc2_207-V2Bbc3_96
3482155,V2Bbc1_20-V2Bbc2_235-V2Bbc3_142
3482156,V2Abc1_273-V2Abc2_51-V2Bbc3_66
3482157,V2Bbc1_108-V2Abc2_339-V2Abc3_237


In [ ]:
barcode_cluster = pd.read_csv(barcode_cluster_path, sep='\t', header=None)
barcode_cluster # 3,482,159 spots. This matches the number of spots expected in the capture area (according to harpy analysis, number of spots included in table). 

,0,1
0,1,81
1,2,97
2,3,134
3,4,177
4,5,201
...,...,...
3482154,3482155,4315749
3482155,3482156,4315769
3482156,3482157,4315776
3482157,3482158,4315781


In [ ]:
Hi, 

I have tried the perl script, but it is not really giving me 

In [18]:
bc_path = f'e:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp99/bc2bc/Exp99.bc'
bc = pd.read_csv(bc_path, sep='\t', names=['barcode_name', 'barcode'], header=None)
bc

,barcode_name,barcode
0,V2Bbc1_359-V2Abc2_4-V2Bbc3_108,L1_80
1,V2Abc1_220-V2Bbc2_349-V2Abc3_56,L1_96
2,V2Abc1_353-V2Bbc2_379-V2Abc3_375,L1_133
3,V2Bbc1_86-V2Abc2_263-V2Bbc3_14,L1_176
4,V2Bbc1_186-V2Bbc2_104-V2Abc3_223,L1_200
...,...,...
3482154,V2Bbc1_71-V2Bbc2_207-V2Bbc3_96,L1_4315748
3482155,V2Bbc1_20-V2Bbc2_235-V2Bbc3_142,L1_4315768
3482156,V2Abc1_273-V2Abc2_51-V2Bbc3_66,L1_4315775
3482157,V2Bbc1_108-V2Abc2_339-V2Abc3_237,L1_4315780


In [22]:
umi_gene = pd.read_csv(r'e:\2024-07-UTBenchmark-SpC\data\processed\BMK_S3000\Exp99\BSTMAtrix\exp99_28Feb2025\umi_gene.tsv.gz', sep='\t', header=None)
umi_gene

,0,1,2,3
0,V2Bbc1_3-V2Bbc2_78-V2Bbc3_87,ACAGCAGC,ENSMUSG00000064341,1
1,V2Bbc1_3-V2Bbc2_78-V2Bbc3_87,GTGATACA,ENSMUSG00000001911,2
2,V2Bbc1_3-V2Bbc2_78-V2Bbc3_87,CAGGGATA,ENSMUSG00000020657,2
3,V2Bbc1_3-V2Bbc2_78-V2Bbc3_87,TTCGCTAGC,ENSMUSG00000045763,1
4,V2Bbc1_3-V2Bbc2_78-V2Bbc3_87,TTGAATAT,ENSMUSG00000030447,2
...,...,...,...,...
138181794,V2Abc1_59-V2Bbc2_352-V2Bbc3_199,CGCCCTAC,ENSMUSG00000060143,1
138181795,V2Abc1_55-V2Bbc2_372-V2Bbc3_139,GGGATTGTCTT,ENSMUSG00000021057,2
138181796,V2Abc1_55-V2Bbc2_372-V2Bbc3_139,AAACACTAAGT,ENSMUSG00000033653,4
138181797,V2Abc1_55-V2Bbc2_372-V2Bbc3_139,TGGAGGCGAGT,ENSMUSG00000039883,2


In [ ]:
# Total number of UMIs
print("Total number of UMIs:", umi_gene[3].sum()) # 311,442,347

Total number of UMIs: 311442347


In [ ]:
print("Number of unique spatial barcodes:", umi_gene[0].nunique()) # 3,295,405 (mathes the total number of spots in table with at least one UMI)

Number of unique spatial barcodes: 3295405


In [ ]:
print("Number of unique UMIs:", umi_gene[1].nunique()) # 21,474,583

Number of unique UMIs: 21474583


In [ ]:
print("Number of unique genes:", umi_gene[2].nunique()) # 28,561 (matches the total number of genes in the table)

Number of unique genes: 28561


In [23]:
path = r'e:\2024-07-UTBenchmark-SpC\data\processed\BMK_S3000\Exp99\BSTMAtrix\exp99_28Feb2025\bc_umi_read.tsv.gz'
bc_umi_read_2 = pd.read_csv(path, sep='\t', header=None)
bc_umi_read_2

,0,1,2
0,V2Bbc1_3-V2Bbc2_78-V2Bbc3_87,ACAGCAGC,1
1,V2Bbc1_3-V2Bbc2_78-V2Bbc3_87,GTGATACA,2
2,V2Bbc1_3-V2Bbc2_78-V2Bbc3_87,CAGGGATA,2
3,V2Bbc1_3-V2Bbc2_78-V2Bbc3_87,TTCGCTAGC,1
4,V2Bbc1_3-V2Bbc2_78-V2Bbc3_87,TTGAATAT,2
...,...,...,...
138181794,V2Abc1_59-V2Bbc2_352-V2Bbc3_199,CGCCCTAC,1
138181795,V2Abc1_55-V2Bbc2_372-V2Bbc3_139,GGGATTGTCTT,2
138181796,V2Abc1_55-V2Bbc2_372-V2Bbc3_139,AAACACTAAGT,4
138181797,V2Abc1_55-V2Bbc2_372-V2Bbc3_139,TGGAGGCGAGT,2


In [ ]:
bc_umi_read_2[2].sum() # 311,442,347 total UMIs

311442347